In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/bank-full.csv', sep=';')

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


# 1. 标准处理

In [3]:
# 二分类变量直接二元编码
df[['default', 'housing', 'loan', 'y']] = df[['default', 'housing', 'loan', 'y']].replace({'yes': 1, 'no': 0}).astype(int)

In [4]:
# 多分类变量进行独热编码
df = pd.get_dummies(df, columns=['job', 'marital', 'education', 'poutcome', 'contact', 'month'], drop_first=True)

In [5]:
df.to_csv('data/processed_data_base.csv', index=False)

# 2. 重点处理

In [6]:
df_test = df.copy()
df_test.head()

,age,default,balance,housing,loan,day,duration,campaign,pdays,previous,...,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep
0,58,0,2143,1,0,5,261,1,-1,0,...,False,False,False,False,False,False,True,False,False,False
1,44,0,29,1,0,5,151,1,-1,0,...,False,False,False,False,False,False,True,False,False,False
2,33,0,2,1,1,5,76,1,-1,0,...,False,False,False,False,False,False,True,False,False,False
3,47,0,1506,1,0,5,92,1,-1,0,...,False,False,False,False,False,False,True,False,False,False
4,33,0,1,0,0,5,198,1,-1,0,...,False,False,False,False,False,False,True,False,False,False


In [7]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 43 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   age                  45211 non-null  int64
 1   default              45211 non-null  int64
 2   balance              45211 non-null  int64
 3   housing              45211 non-null  int64
 4   loan                 45211 non-null  int64
 5   day                  45211 non-null  int64
 6   duration             45211 non-null  int64
 7   campaign             45211 non-null  int64
 8   pdays                45211 non-null  int64
 9   previous             45211 non-null  int64
 10  y                    45211 non-null  int64
 11  job_blue-collar      45211 non-null  bool 
 12  job_entrepreneur     45211 non-null  bool 
 13  job_housemaid        45211 non-null  bool 
 14  job_management       45211 non-null  bool 
 15  job_retired          45211 non-null  bool 
 16  job_self-employed    45211 non-nu

1. 对 balance、campaign、previous 尝试对数变换。对于余额，由于存在负数，使用保留正负号的对数变换

In [8]:
df_test['balance'] = np.sign(df_test['balance']) * np.log1p(np.abs(df_test['balance']))
df_test['campaign'] = np.log1p(df_test['campaign'])
df_test['previous'] = np.log1p(df_test['previous'])

2. 对于pdays，区分客户以前是否被联系过，以及联系过的客户距离上次联系有多少天。


In [9]:
df_test['pre_contact'] = (df_test['pdays'] != -1).astype(int)
df_test['pdays'] = np.where(df_test['pdays'] == -1, np.log1p(df_test['pdays']).clip(lower=0), 0)

d:\Python312\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


3. 对于年龄可以考虑分箱

In [10]:
df_test['age_group'] = pd.cut(df_test['age'], bins=[0, 25, 35, 45, 55, 65, 100], labels=['0-25', '26-35', '36-45', '46-55', '56-65', '65+'])
df_test = pd.get_dummies(df_test, columns=['age_group'], drop_first=True)

In [11]:
df['y'].value_counts()

y
0    39922
1     5289
Name: count, dtype: int64

In [12]:
df_test.to_csv('data/processed_data.csv', index=False)